# SwinIR SEM 微调 — 正式训练

GPU: 2×T4 | Checkpoint: 10k iter | 目标: 70k iter

**策略**：
1. 先跑 200 iter 试运行（验证训练能启动、ETA 合理）
2. 确认无误后，跑完整训练
3. 训练完成后下载 output 中的模型和日志

## 0. 环境准备（与 check.ipynb 相同）

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, GPUs: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        mem = getattr(p, 'total_memory', getattr(p, 'total_mem', 0))
        print(f'  GPU {i}: {p.name} — {mem/1e9:.1f} GB')

In [ ]:
import os, shutil
WORK_DIR = '/kaggle/working'
os.chdir(WORK_DIR)

# 克隆仓库
if not os.path.exists('BasicSR'):
    !git clone https://github.com/Log-Dog012/BasicSR.git
    !cd BasicSR && git checkout cuda-sem-finetune

# 安装依赖
os.chdir('BasicSR')
!pip install -r requirements.txt -q
!pip install -e . -q
!pip install lpips timm -q
print('✅ 环境就绪')

In [ ]:
# 复制模型文件
MODEL_INPUT = '/kaggle/input/models/logdog012/swinir-finetune/pytorch/default/1'
swinir_zoo = os.path.join(WORK_DIR, 'BasicSR', 'SwinIR', 'model_zoo')
exp_dir = os.path.join(WORK_DIR, 'BasicSR', 'experiments', 'finetune_SwinIR_SRx4_SEM')

# 预训练权重
os.makedirs(swinir_zoo, exist_ok=True)
for root, dirs, files in os.walk(MODEL_INPUT):
    for f in files:
        if 'classicalSR' in f and f.endswith('.pth'):
            shutil.copy2(os.path.join(root, f), os.path.join(swinir_zoo, f))
            print(f'✅ 预训练: {f}')
            break

# Checkpoint
for name in ['net_g_10000.pth', '10000.state']:
    for root, dirs, files in os.walk(MODEL_INPUT):
        if name in files:
            dst_dir = os.path.join(exp_dir, 'models' if 'pth' in name else 'training_states')
            os.makedirs(dst_dir, exist_ok=True)
            shutil.copy2(os.path.join(root, name), os.path.join(dst_dir, name))
            print(f'✅ {name}')
            break
print('✅ 模型文件就绪')

## 1. 试运行（200 iter，确认训练正常）

In [ ]:
import yaml

cfg_path = 'options/train/SwinIR/finetune_SwinIR_SRx4_SEM.yml'
with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

# 试运行：不改 total_iter（checkpoint 已在 10000），只改打印和保存频率
# 训练会从 10000 继续跑，每 50 iter 打印，跑 200 iter 后停止观察
cfg['train']['total_iter'] = 10200      # 10000 + 200 = 试跑 200 iter
cfg['logger']['print_freq'] = 50
cfg['logger']['save_checkpoint_freq'] = 50000  # 不存 checkpoint，节省时间
cfg['val']['val_freq'] = 50000         # 不验证
cfg['auto_resume'] = True

print(f'模型权重: {cfg["path"]["pretrain_network_g"]}')
print(f'训练 HR: {cfg["datasets"]["train"]["dataroot_gt"]}')
print(f'num_gpu={cfg["num_gpu"]}, batch={cfg["datasets"]["train"]["batch_size_per_gpu"]}')
print(f'total_iter={cfg["train"]["total_iter"]} (从 10000 续训 200 iter)')

# 保存试运行配置
test_cfg_path = 'options/train/SwinIR/finetune_test_run.yml'
with open(test_cfg_path, 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
print(f'试运行配置: {test_cfg_path}')

In [ ]:
# 试运行 200 iter — DDP 模式
print('开始试运行 (200 iter, DDP 2xT4)...')
print('='*50)
!torchrun --nproc_per_node=2 --master_port=4321 -m basicsr.train -opt options/train/SwinIR/finetune_test_run.yml --launcher pytorch

In [ ]:
# 检查试运行结果
log_dir = os.path.join(exp_dir)
logs = sorted([f for f in os.listdir(log_dir) if f.endswith('.log')])
if logs:
    latest_log = os.path.join(log_dir, logs[-1])
    with open(latest_log, 'r') as f:
        lines = f.readlines()
    # 打印最后 10 行
    print(f'日志: {logs[-1]}')
    print('='*60)
    for line in lines[-10:]:
        print(line.rstrip())
    
    # 检查是否有报错
    errors = [l for l in lines if 'Error' in l or 'Traceback' in l]
    if errors:
        print(f'\n❌ 发现 {len(errors)} 个错误！')
    else:
        # 提取 ETA
        eta_lines = [l for l in lines if 'eta' in l]
        if eta_lines:
            print(f'\n✅ 试运行成功！最后的 ETA: {eta_lines[-1].strip()}')
else:
    print('❌ 未找到日志文件')

## 2. 正式训练

试运行确认无误后，运行此 cell。训练 70k iter，预计 10-12 小时（2×T4）。

**注意**：Kaggle Notebook 有 12 小时限制。如果超时，`auto_resume: true` 会在下次运行时自动恢复。

In [ ]:
# 恢复正式配置
with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print('正式训练配置:')
print(f'  total_iter: {cfg["train"]["total_iter"]}')
print(f'  lr: {cfg["train"]["optim_g"]["lr"]}')
print(f'  milestones: {cfg["train"]["scheduler"]["milestones"]}')
print(f'  num_gpu: {cfg["num_gpu"]}, batch: {cfg["datasets"]["train"]["batch_size_per_gpu"]}')
print(f'  auto_resume: {cfg["auto_resume"]}')

In [ ]:
# 正式训练 — 使用 DDP（torchrun）替代 DataParallel
# DP 是把模型复制到多卡，每步 CPU 汇聚梯度，极慢
# DDP 是每卡独立计算，NCCL 高速通信，快 2-3x
print('开始正式训练 (70k iter, DDP 2xT4)...')
print('='*50)
!torchrun --nproc_per_node=2 --master_port=4321 -m basicsr.train -opt options/train/SwinIR/finetune_SwinIR_SRx4_SEM.yml --launcher pytorch

## 3. 结果分析

In [ ]:
# 提取所有验证结果
logs = sorted([f for f in os.listdir(exp_dir) if f.endswith('.log')])
if logs:
    latest_log = os.path.join(exp_dir, logs[-1])
    with open(latest_log, 'r') as f:
        content = f.read()
    
    # 提取验证 PSNR
    import re
    val_matches = re.findall(r'psnr:\s+([\d.]+)\s+Best:\s+([\d.]+)\s+@\s+(\d+)', content)
    
    if val_matches:
        print('验证结果:')
        print(f'{"Iter":<10} {"PSNR":<10} {"Best PSNR":<12}')
        print('-' * 35)
        best_psnr = 0
        for psnr_val, best_val, iter_val in val_matches:
            marker = ' ←' if float(psnr_val) == float(best_val) else ''
            print(f'{iter_val:<10} {psnr_val:<10} {best_val:<12}{marker}')
    
    # 最后几行
    lines = content.strip().split('\n')
    print(f'\n日志最后 5 行:')
    for line in lines[-5:]:
        print(line)
else:
    print('未找到日志')

## 4. 保存结果到 output

训练完成后，output 目录中的文件会被保存为 Kaggle Dataset，可以下载。

In [ ]:
# /kaggle/working 整个目录在 Save and Run All 后会自动保存为 output
# 不需要手动复制，直接查看训练产物即可

print('训练产物位置:')
print(f'  模型权重: {models_dir}')
print(f'  训练状态: {states_dir}')

# 列出模型文件
print(f'\n模型文件:')
for f in sorted(os.listdir(models_dir)):
    size = os.path.getsize(os.path.join(models_dir, f)) / 1e6
    print(f'  {f} ({size:.1f}MB)')

# 列出训练状态文件
print(f'\n训练状态:')
for f in sorted(os.listdir(states_dir)):
    size = os.path.getsize(os.path.join(states_dir, f)) / 1e6
    print(f'  {f} ({size:.1f}MB)')

# 列出日志
print(f'\n日志文件:')
for f in sorted(os.listdir(exp_dir)):
    if f.endswith('.log'):
        size = os.path.getsize(os.path.join(exp_dir, f)) / 1e6
        print(f'  {f} ({size:.1f}MB)')


In [ ]:
# Kaggle Save and Run All 会自动保存 /kaggle/working 下所有内容
# 右侧面板 → Data → 下载即可

working_size = sum(
    os.path.getsize(os.path.join(r, f))
    for r, _, files in os.walk('/kaggle/working')
    for f in files
)
print(f'working 目录总大小: {working_size/1e9:.2f} GB')
print(f'训练完成！Save and Run All 后，右侧面板 → Data → 下载。')